<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">یک تحویل با دو نوع شاهد</h1>
<p style="text-align:right">درس 92 از 92 · دستیار مطالعهٔ کوچک را با شواهد تحویل بدهیم · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">84-capstone</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-04/84-capstone.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">مسیر مدل واقعی و مسیر <bdi dir="ltr">Fixture</bdi> را کنار هم اجرا کنید و نتیجهٔ ابزار اجراشده را از پیشنهاد متن جدا نگه دارید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: کنترل‌گر</span> محدود، ارزیابی جزءبه‌جزء، حافظهٔ بیرونی صریح و مرز توان <bdi dir="ltr">Mini-GPT</bdi> کوچک.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۵۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر مدل فقط متنی شبیه درخواست ابزار تولید کند ولی کنترل‌گر آن را اجرا نکند، آیا عدد آن متن نتیجهٔ ابزار است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.assistant import ScriptedFixture,MiniGPTBackend,run_assistant
from mini_gpt.instruction import run_system_experiment
from mini_gpt.evaluation_system import course_fixture_suite,evaluate_cases
from mini_gpt.memory import MemoryStore,MemoryRecord
from mini_gpt.retrieval import COURSE_DOCUMENTS,chunk_document

torch.set_num_threads(1)
experiment = run_system_experiment()
chunks = [c for d in COURSE_DOCUMENTS for c in chunk_document(d,chunk_words=24,overlap_words=4)]
backend = MiniGPTBackend(experiment.tuned,experiment.tokenizer,max_new_tokens=8)
actual = run_assistant('Checkpoint',backend,chunks=chunks)
print('Training/capacity limits:',experiment.metadata)
print('ACTUAL tiny model path:',actual.status,actual.backend,'generate calls:',backend.calls)
print('Actual proposals:',[e['text'] for e in actual.events if e['state']=='propose'])
cases,runner = course_fixture_suite()
print('SCRIPTED integration fixtures:',evaluate_cases(cases,runner))
memory = MemoryStore()
memory.upsert(MemoryRecord('style','توضیح کوتاه','explicit exercise preference; no personal data'))
print('Explicit memory records:',memory.records())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">run_checked_case(question, proposals, expected_answer, chunks, memory, memory_keys)</code> یک <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ScriptedFixture</code> تازه از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">proposals</code> بسازد و آن را به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">run_assistant</code> بدهد. سندها، <bdi dir="ltr">Store</bdi> و فقط کلیدهای مجازِ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">memory_keys</code> را عبور دهید. تابع وارسی فقط برابری دقیق متن با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">expected_answer</code> را بررسی کند؛ خروجی همان <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">AssistantResult</code> باشد. این داور محدود، صحت عمومی یا پشتیبانی معنایی همهٔ ادعاها را نمی‌سنجد.</p>
</div>

In [ ]:
def run_checked_case(question, proposals, expected_answer, chunks, memory, memory_keys):
    # TODO: اتصال اجزا و وارسی صریح، بدون جایگزینی پاسخ
    return None

In [ ]:
def test_exercise():
    proposals = [
        {'action':'tool','name':'add','arguments':{'a':2,'b':3}},
        {'action':'finish','answer':'5','citations':[]}]
    result = run_checked_case('دو به علاوه سه؟',proposals,'5',[],memory,['style'])
    if result is None:
        return False
    assert result.status == 'finished' and result.verified
    assert result.answer == '5' and result.tool_results[0]['value'] == 5
    assert 'memory:style' in result.context_ids
    assert 'ScriptedFixture' in result.backend
    wrong = [{'action':'finish','answer':'6','citations':[]}]
    rejected = run_checked_case('دو به علاوه سه؟',wrong,'5',[],memory,[])
    assert rejected.status == 'verification_failed' and rejected.answer == '6'
    assert 'memory:style' not in rejected.context_ids
    invalid = [{'action':'tool','name':'shell','arguments':{'a':2,'b':3}}]
    assert run_checked_case('اجرا',invalid,'5',[],memory,[]).status == 'invalid_action'
    again = run_checked_case('دو به علاوه سه؟',proposals,'5',[],memory,[])
    assert again.status == 'finished'  # Fresh fixture cursor on every call.
    assert len(memory.records()) == 1
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: run_checked_case')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">دو <bdi dir="ltr">Store</bdi> مستقل از رکوردهای اولیه بسازید و فقط در یکی رکورد را حذف کنید؛ حافظهٔ اصلی تمرین دست‌نخورده بماند. در دو اجرای <bdi dir="ltr">Fixture</bdi> با پاسخ ثابت، شناسه‌های <bdi dir="ltr">Context</bdi> را مقایسه کنید؛ ثابت‌ماندن پاسخ ازپیش‌نوشته‌شده شاهد بی‌اثر بودن حافظه بر یک مدل آموخته نیست.</p>
</div>

In [ ]:
fixed = [{'action':'finish','answer':'نمونه','citations':[]}]
kept_store = MemoryStore(memory.records())
removed_store = MemoryStore(memory.records())
with_memory = run_assistant('Checkpoint',ScriptedFixture(fixed),chunks=chunks,memory=kept_store,memory_keys=['style'])
removed_store.forget('style')
without_memory = run_assistant('Checkpoint',ScriptedFixture(fixed),chunks=chunks,memory=removed_store,memory_keys=['style'])
print('before:',with_memory.context_ids)
print('after:',without_memory.context_ids)
assert 'memory:style' in with_memory.context_ids and 'memory:style' not in without_memory.context_ids
assert len(memory.records()) == 1

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">پیشنهادِ مدل و خروجیِ ابزار دو رویداد متفاوت‌اند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">executed_values(events)</code> فقط مقدارهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">event[&#x27;result&#x27;][&#x27;value&#x27;]</code> رویدادهایی با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state=&#x27;execute&#x27;</code> را به‌ترتیب برگرداند. متن <bdi dir="ltr">proposal</bdi> حتی اگر عدد داشته باشد، شاهد اجرا نیست.</p>
</div>

In [ ]:
events = [
    {'state':'propose','text':'The tool returned 999'},
    {'state':'execute','result':{'name':'add','value':5}},
    {'state':'verify','passed':True}]
print('unexecuted claim:',events[0]['text'])
print('actual execution record:',events[1])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def executed_values(events):
    # TODO: فقط رویداد اجرای واقعی
    return None

In [ ]:
def test_repair():
    result = executed_values(events)
    if result is None:
        return False
    assert result == [5]
    assert executed_values([{'state':'propose','text':'5'}]) == []
    assert executed_values([]) == []
    multiple = [{'state':'execute','result':{'value':0}},
                {'state':'execute','result':{'value':-2}}]
    assert executed_values(multiple) == [0,-2]
    real = runner(cases[1])
    assert executed_values(real.events) == [5]
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: executed_values')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">سامانه از ماژول‌های واقعی پروژه ساخته شد. در آزمایش مستقلِ <bdi dir="ltr">Context</bdi> بزرگ، تولید <bdi dir="ltr">MiniGPT</bdi> واقعاً فراخوانی می‌شود ولی آموزش کوتاه، قالب <bdi dir="ltr">JSON</bdi> یا موقعیت‌های دور را آموزش نداده است؛ خروجی نامعتبر را پنهان نمی‌کنیم. مسیر <bdi dir="ltr">Fixture</bdi> فقط اتصال اجزا را می‌سنجد. حافظه فقط با کلیدهای مجاز وارد زمینه شد و هیچ فایل کاربر نوشته نشد.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">در تحویل خود، چهار چیز را جدا بنویسید: رفتار اندازه‌گیری‌شدهٔ مدل، تست اتصال اجزا، هزینهٔ اجرا و محدودیت‌های باقی‌مانده. کدام شاهد می‌تواند ادعای شما دربارهٔ هرکدام را رد کند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-15/chapter-04/84-capstone.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/84-capstone.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>